# Imports

In [ ]:
from openmeteo_requests import Client as OpenMeteoClient
from requests_cache import CachedSession
from retry_requests import retry
import pandas as pd 
import requests
import datetime
import time

# Adding weather informations by date

## Constants

In [ ]:
BASE_URL_WEATHER = 'https://archive-api.open-meteo.com/v1/archive'
BASE_URL_IP = 'http://ip-api.com/batch?fields=query,status,lat,lon'

PATH_SPOTIFY = '../../data/2_processed/final_df.csv'
PATH_IPS = '../../data/2_processed/ip_addresses.csv'
PATH_WEATHER = '../../data/2_processed/weather.csv'

## Methods

### Address/IP

In [ ]:
def get_address_by_list_ips(list_ips):
    url = BASE_URL_IP
    dict_address = []
    
    try:
        response = requests.post(url, json=list_ips)
        response.raise_for_status()
        results = response.json()
        
        for info in results:
            if info.get('status') == 'success':
                ip = info.get('query')
                lat, lon = float(info.get('lat')), float(info.get('lon'))
                dict_address.append({ip:f'{lat:.1f},{lon:.1f}'})
            else:
                print(f"Not able to locate IP: {info.get('query')}")
        
    except requests.exceptions.RequestException as e:
        print(f'Request error: {e}')
        for i in list_ips:
            dict_address.append({i:None})

    return dict_address

In [ ]:
def set_address_by_ip(df):
    all_ips = list(set(df.ip_addr))
    all_address = []
    batch_ips = []
    count_request = 0
    for i in all_ips:
        batch_ips.append(i)
        if (len(batch_ips) >= 99):
            all_address.extend(get_address_by_list_ips(batch_ips))
            batch_ips = []
            count_request += 1
            if (count_request >= 14):
                print('Wait 1 minute before making the next call.')
                time.sleep(60)
                count_request = 0

    return all_address

### Weather

In [ ]:
def get_weather_history(batch_lat, batch_lon, data_inicio, data_fim):
    cache_session = CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = OpenMeteoClient(session=retry_session)

    params = {
        'latitude': batch_lat, 'longitude': batch_lon,
        'start_date': data_inicio, 'end_date': data_fim,
        'hourly': ['temperature_2m', 'precipitation'],
    }

    responses = openmeteo.weather_api(BASE_URL_WEATHER, params=params)

    all_dfs = []
    for r in responses:
        hourly = r.Hourly()
        dates = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive='left',
        )
        all_dfs.append(pd.DataFrame({
            'date': dates,
            'temperature': hourly.Variables(0).ValuesAsNumpy(),
            'precipitation': hourly.Variables(1).ValuesAsNumpy(),
            'lat_long': f'{r.Latitude():.1f},{r.Longitude():.1f}',
        }))

    return pd.concat(all_dfs)

In [ ]:
def get_all_weather(df):
    date_start = df.iloc[0].ts.split('T')[0]
    date_end = df.iloc[-1].ts.split('T')[0]

    DEFAULT_COORDS = (-23.5471, -46.6372)  # São Paulo
    lats, lons = zip(*(
        tuple(map(float, addr.split(','))) if str(addr).upper() != 'NAN' else DEFAULT_COORDS
        for addr in set(df['lat_long'])
    ))

    all_weather, req_count = [], 0

    for i in range(0, len(lats), 50):
        batch_lat, batch_lon = list(lats[i:i+50]), list(lons[i:i+50])
        all_weather.append(get_weather_history(batch_lat, batch_lon, date_start, date_end))

        if req_count >= 19:
            req_count = 0
            print('Waiting 1 minute for more api calls.')
            time.sleep(60)
        else:
            req_count += 1

    return pd.concat(all_weather)


In [ ]:
def optimize_weather_merge(main_df, weather_df):
    # Ensure 'ts' is datetime
    main_df["ts"] = pd.to_datetime(main_df["ts"])

    # Build temporary join keys
    main_df["date"] = main_df["ts"].dt.strftime("%Y-%m-%d")
    main_df["hour"] = main_df["ts"].dt.hour

    # Left-join weather data on location + date + hour
    result_df = (
        main_df
        .merge(
            weather_df[["lat_long", "date", "hour", "temperature", "precipitation"]],
            on=["lat_long", "date", "hour"],
            how="left",
        )
        .drop(columns=["date", "hour"])  # drop temporary join keys
    )

    return result_df

## Creating address df

In [ ]:
final_df = pd.read_csv('final_df.csv')

In [ ]:
address_df = final_df.copy()
all_address = set_address_by_ip(address_df)

In [ ]:
df = pd.DataFrame([
    {'ip': list(d.keys())[0], 'lat_long': list(d.values())[0]} 
    for d in all_address
])
ips_address_df = pd.merge(address_df, df, how='left', left_on='ip_addr', right_on='ip').drop(columns='ip')
ips_address_df

In [ ]:
ips_address_df.to_csv(PATH_IPS)

## Creating weather df

In [ ]:
weather_df = pd.read_csv(PATH_IPS).drop(columns='Unnamed: 0')
display(weather_df)

In [ ]:
just_weather_df = get_all_weather(weather_df)

In [ ]:
all_datetimes = just_weather_df['date']
all_dates, all_hours = [], []
for i in all_datetimes:
    all_dates.append(i.strftime('%Y-%m-%d'))
    all_hours.append(i.hour)

just_weather_df['date'] = all_dates
just_weather_df['hour'] = all_hours
just_weather_df

In [ ]:
just_weather_df.to_csv(PATH_WEATHER)
just_weather_df

## Creating final df with weather

In [ ]:
just_weather_df = pd.read_csv(PATH_WEATHER)

final_weather_df = optimize_weather_merge(weather_df, just_weather_df)
final_weather_df

In [ ]:
final_weather_df.to_csv(PATH_SPOTIFY)